In [1]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    preprocessing_function=None,   # EfficientNet uses rescale
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/train",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)

val_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/val",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical"
)


Found 4835 images belonging to 3 classes.
Found 1031 images belonging to 3 classes.


In [3]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [4]:
for layer in base_model.layers:
    layer.trainable = False


In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(3, activation="softmax")(x)   # covid, normal, pneumonia

model = Model(inputs=base_model.input, outputs=output)

In [6]:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [7]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=val_gen
)


Epoch 1/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 69s 216ms/step - accuracy: 0.3371 - loss: 1.1194 - val_accuracy: 0.3356 - val_loss: 1.1043
Epoch 2/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 71s 236ms/step - accuracy: 0.3187 - loss: 1.1051 - val_accuracy: 0.3288 - val_loss: 1.1005
Epoch 3/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 79s 259ms/step - accuracy: 0.3245 - loss: 1.1024 - val_accuracy: 0.3356 - val_loss: 1.0988
Epoch 4/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 81s 267ms/step - accuracy: 0.3330 - loss: 1.0996 - val_accuracy: 0.3356 - val_loss: 1.0990
Epoch 5/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 97s 321ms/step - accuracy: 0.3311 - loss: 1.0997 - val_accuracy: 0.3356 - val_loss: 1.0986
Epoch 6/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 99s 327ms/step - accuracy: 0.3299 - loss: 1.0995 - val_accuracy: 0.3356 - val_loss: 1.0987
Epoch 7/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 92s 302ms/step - accuracy: 0.3251 - loss: 1.0989 - val_accuracy: 0.3356 - val_loss: 1.0986
Epoch 8/15
303/303 ━━━━━━━━━━━━━━━━━━━━ 101s 332ms/step - accuracy: 0.3404 - loss: 

In [ ]:
test_gen = val_datagen.flow_from_directory(
    "/Users/kpvarma/PycharmProjects/Chest_X_Rays_Pnemonia_3/Dataset/test",
    target_size=(224,224),
    batch_size=16,
    class_mode="categorical",
    shuffle=False
)

model.evaluate(test_gen)


Found 1036 images belonging to 3 classes.
51/65 ━━━━━━━━━━━━━━━━━━━━ 2s 194ms/step - accuracy: 0.2098 - loss: 1.1022

In [ ]:
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)


Epoch 1/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 13694s 43s/step - accuracy: 0.3349 - loss: 1.1321 - val_accuracy: 0.3385 - val_loss: 1.1019
Epoch 2/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 10395s 34s/step - accuracy: 0.3253 - loss: 1.1223 - val_accuracy: 0.3346 - val_loss: 1.0957
Epoch 3/10
303/303 ━━━━━━━━━━━━━━━━━━━━ 14217s 47s/step - accuracy: 0.3456 - loss: 1.1129 - val_accuracy: 0.4054 - val_loss: 1.0938
Epoch 4/10
145/303 ━━━━━━━━━━━━━━━━━━━━ 1:31:56 35s/step - accuracy: 0.3486 - loss: 1.1125